# ClusterJudge 2 – Modular Notebook for Comparing Clusterings

This notebook follows the specification in `prompt_main2.txt`.
Every analysis cell exposes functions that later cells can reuse, and every saved figure is paired with a CSV file that the plotting function can read back to reproduce the figure.


## Cell 1 – Setup


In [ ]:
# !pip install -q networkx scipy seaborn

import itertools
import logging
import math
import os
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from matplotlib.patches import ConnectionPatch
from scipy.linalg import orthogonal_procrustes

warnings.filterwarnings("ignore")

SETUP_SEED = 42  # Higher changes every downstream random draw; lower just picks a different reproducible world.
FIGURE_DIR = "figures/main2"  # Change this to redirect outputs; nesting deeper keeps runs more organized.
LOG_PATH = "logs/main2.log"  # Change this to separate logs across experiments; reuse to append a single experiment trail.
DEFAULT_DPI = 180  # Higher makes raster figures crisper but larger; lower writes smaller JPG files.


def seed_everything(seed: int) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_parent(path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def save_dataframe(df: pd.DataFrame, path: str | Path) -> Path:
    path = ensure_parent(path)
    df.to_csv(path, index=False)
    return path


def save_figure_bundle(fig, stem: str | Path, dpi: int = DEFAULT_DPI) -> tuple[Path, Path]:
    stem = Path(stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    jpg_path = stem.with_suffix(".jpg")
    pdf_path = stem.with_suffix(".pdf")
    fig.savefig(jpg_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    return jpg_path, pdf_path


def marker_for_cluster(cluster_id: int) -> str:
    markers = ["^", "o", "v", "s", "D", "P", "X"]
    return markers[int(cluster_id) % len(markers)]


def palette_color(index: int, palette_name: str = "deep") -> tuple[float, float, float]:
    palette = sns.color_palette(palette_name, 10)
    return palette[int(index) % len(palette)]


def canonical_pair(i: int, j: int) -> tuple[int, int]:
    return (int(i), int(j)) if int(i) <= int(j) else (int(j), int(i))


def pairwise_consistency_metrics(true_labels, pred_labels) -> dict:
    true_labels = np.asarray(true_labels)
    pred_labels = np.asarray(pred_labels)

    tp = tn = fp = fn = 0
    for i in range(len(true_labels)):
        for j in range(i + 1, len(true_labels)):
            true_same = int(true_labels[i] == true_labels[j])
            pred_same = int(pred_labels[i] == pred_labels[j])
            if true_same == 1 and pred_same == 1:
                tp += 1
            elif true_same == 0 and pred_same == 0:
                tn += 1
            elif true_same == 0 and pred_same == 1:
                fp += 1
            else:
                fn += 1

    total = tp + tn + fp + fn
    positive_total = tp + fn
    negative_total = tn + fp
    tpr = tp / positive_total if positive_total else 0.0
    tnr = tn / negative_total if negative_total else 0.0
    balanced_accuracy = 0.5 * (tpr + tnr)

    return {
        "pair_accuracy": (tp + tn) / total if total else 0.0,
        "balanced_accuracy": balanced_accuracy,
        "true_positive_rate": tpr,
        "true_negative_rate": tnr,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def assignment_dict_to_array(assignment: dict[int, int], n_points: int) -> np.ndarray:
    return np.array([int(assignment.get(i, 0)) for i in range(n_points)], dtype=int)


def greedy_maxcut_partition(
    graph: nx.Graph,
    n_clusters: int,
    n_restarts: int,
    max_iter: int,
    seed: int,
) -> tuple[dict[int, int], float]:
    rng = np.random.default_rng(seed)
    nodes = list(graph.nodes())
    if not nodes:
        return {}, 0.0

    def local_score(assignment: dict[int, int], node: int, cluster_id: int) -> float:
        score = 0.0
        for nbr, edge_data in graph[node].items():
            if assignment[nbr] != cluster_id:
                score += float(edge_data.get("weight", 1.0))
        return score

    best_assignment = {node: 0 for node in nodes}
    best_value = -np.inf

    for _ in range(n_restarts):
        assignment = {node: int(rng.integers(0, n_clusters)) for node in nodes}
        for _ in range(max_iter):
            improved = False
            order = list(nodes)
            rng.shuffle(order)
            for node in order:
                current_cluster = assignment[node]
                best_cluster = current_cluster
                best_cluster_score = local_score(assignment, node, current_cluster)
                for candidate in range(n_clusters):
                    if candidate == current_cluster:
                        continue
                    candidate_score = local_score(assignment, node, candidate)
                    if candidate_score > best_cluster_score:
                        best_cluster = candidate
                        best_cluster_score = candidate_score
                if best_cluster != current_cluster:
                    assignment[node] = best_cluster
                    improved = True
            if not improved:
                break

        cut_value = 0.0
        for u, v, edge_data in graph.edges(data=True):
            if assignment[u] != assignment[v]:
                cut_value += float(edge_data.get("weight", 1.0))

        if cut_value > best_value:
            best_value = cut_value
            best_assignment = dict(assignment)

    return best_assignment, float(best_value)


def hierarchical_mincut_partition(graph: nx.Graph, n_clusters: int) -> dict[int, int]:
    if graph.number_of_nodes() == 0:
        return {}

    groups = [list(graph.nodes())]

    while len(groups) < n_clusters:
        candidates = []
        for group in groups:
            subgraph = graph.subgraph(group).copy()
            if len(group) > 1:
                candidates.append((group, subgraph))
        if not candidates:
            break

        target_group, target_subgraph = max(candidates, key=lambda item: item[1].number_of_edges())
        if target_subgraph.number_of_edges() == 0:
            midpoint = len(target_group) // 2
            split_a = target_group[:midpoint]
            split_b = target_group[midpoint:]
        else:
            try:
                _, partition = nx.stoer_wagner(target_subgraph, weight="weight")
                split_a = list(partition[0])
                split_b = list(partition[1])
            except Exception:
                midpoint = len(target_group) // 2
                split_a = target_group[:midpoint]
                split_b = target_group[midpoint:]

        groups.remove(target_group)
        if split_a:
            groups.append(split_a)
        if split_b:
            groups.append(split_b)
        if not split_a or not split_b:
            break

    assignment = {}
    for cluster_id, group in enumerate(groups):
        for node in group:
            assignment[int(node)] = int(cluster_id)
    return assignment


def plot_cluster_scatter_from_csv(
    csv_path: str | Path,
    stem: str | Path,
    title: str,
    palette_name: str = "deep",
) -> tuple[Path, Path]:
    df = pd.read_csv(csv_path)
    palette = sns.color_palette(palette_name, max(int(df["inferred_cluster"].max()) + 1, 3))
    fig, ax = plt.subplots(figsize=(6.0, 4.8))
    for cluster_id in sorted(df["inferred_cluster"].unique()):
        mask = df["inferred_cluster"] == cluster_id
        ax.scatter(
            df.loc[mask, "x"],
            df.loc[mask, "y"],
            facecolors="none",
            edgecolors=[palette[int(cluster_id) % len(palette)]],
            marker=marker_for_cluster(cluster_id),
            linewidths=1.8,
            s=90,
            label=f"Inferred cluster {int(cluster_id)}",
        )
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend(frameon=False, fontsize=8)
    sns.despine(ax=ax)
    fig.tight_layout()
    paths = save_figure_bundle(fig, stem)
    plt.show()
    plt.close(fig)
    return paths


def configure_environment(seed: int, figure_dir: str, log_path: str) -> dict:
    seed_everything(seed)
    Path(figure_dir).mkdir(parents=True, exist_ok=True)
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        filename=log_path,
        level=logging.INFO,
        format="%(asctime)s %(levelname)s %(message)s",
        force=True,
    )
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
    env = {
        "seed": seed,
        "figure_dir": figure_dir,
        "log_path": log_path,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "dtype": torch.float32,
    }
    logging.info("Configured environment: %s", env)
    return env


MAIN2_ENV = configure_environment(SETUP_SEED, FIGURE_DIR, LOG_PATH)
MAIN2_STATE = {}

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {MAIN2_ENV['device']}")
print(f"Figure dir      : {MAIN2_ENV['figure_dir']}")
print(f"Log path        : {MAIN2_ENV['log_path']}")


## Cell 2 – Dataset Generation


In [ ]:
DATASET_POINTS_PER_CLUSTER = 10  # Higher gives denser clusters; lower makes every random sample look more irregular.
DATASET_SEED = 42  # Higher only changes the sampled coordinates; keeping it fixed keeps all later cells comparable.
DATASET_CLUSTER_PARAMS = [
    {"mean_x": 0.0, "mean_y": 0.0, "std": 1.0, "label": 0},
    {"mean_x": 2.0, "mean_y": 2.0, "std": 0.3, "label": 1},
    {"mean_x": -5.0, "mean_y": -5.0, "std": 0.5, "label": 2},
]  # Moving means closer or increasing std makes the clustering task harder; spreading means apart makes it easier.
DATASET_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/ground_truth.csv"  # Change this to save the reusable plot data elsewhere.
DATASET_FIGURE_STEM = f"{MAIN2_ENV['figure_dir']}/ground_truth"  # Change this stem to rename both JPG and PDF together.


def generate_gaussian_dataset(
    cluster_params: list[dict],
    points_per_cluster: int,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    point_idx = 0
    for spec in cluster_params:
        mean = np.array([spec["mean_x"], spec["mean_y"]], dtype=float)
        std = float(spec["std"])
        label = int(spec["label"])
        samples = rng.normal(loc=mean, scale=std, size=(points_per_cluster, 2))
        for x_coord, y_coord in samples:
            rows.append(
                {
                    "point_idx": point_idx,
                    "x": float(x_coord),
                    "y": float(y_coord),
                    "true_cluster": label,
                    "marker": marker_for_cluster(label),
                }
            )
            point_idx += 1
    return pd.DataFrame(rows)


def plot_ground_truth_from_csv(csv_path: str | Path, stem: str | Path) -> tuple[Path, Path]:
    df = pd.read_csv(csv_path)
    palette = sns.color_palette("deep", max(int(df["true_cluster"].max()) + 1, 3))
    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    for cluster_id in sorted(df["true_cluster"].unique()):
        mask = df["true_cluster"] == cluster_id
        ax.scatter(
            df.loc[mask, "x"],
            df.loc[mask, "y"],
            facecolors="none",
            edgecolors=[palette[int(cluster_id) % len(palette)]],
            marker=marker_for_cluster(cluster_id),
            linewidths=1.8,
            s=90,
            label=f"Cluster {int(cluster_id)}",
        )
    ax.set_title("Ground-Truth Dataset")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend(frameon=False, fontsize=8)
    sns.despine(ax=ax)
    fig.tight_layout()
    paths = save_figure_bundle(fig, stem)
    plt.show()
    plt.close(fig)
    return paths


def run_dataset_cell(
    cluster_params: list[dict],
    points_per_cluster: int,
    seed: int,
    csv_path: str | Path,
    figure_stem: str | Path,
) -> dict:
    points_df = generate_gaussian_dataset(cluster_params, points_per_cluster, seed)
    save_dataframe(points_df, csv_path)
    plot_ground_truth_from_csv(csv_path, figure_stem)
    logging.info("Dataset cell completed with %d points", len(points_df))
    return {
        "points_df": points_df,
        "csv_path": str(csv_path),
        "figure_stem": str(figure_stem),
    }


MAIN2_STATE["dataset"] = run_dataset_cell(
    cluster_params=DATASET_CLUSTER_PARAMS,
    points_per_cluster=DATASET_POINTS_PER_CLUSTER,
    seed=DATASET_SEED,
    csv_path=DATASET_CSV_PATH,
    figure_stem=DATASET_FIGURE_STEM,
)

MAIN2_STATE["dataset"]["points_df"].head()


## Cell 3 – Judgement Generation


In [ ]:
JUDGEMENT_COUNT = 20  # Higher gives better graph coverage; lower keeps the judge signal much noisier.
JUDGEMENT_TEMPERATURE = 1.0  # Higher makes choices closer to random; lower makes the nearest pair dominate.
JUDGEMENT_SEED = 42  # Higher changes which triples are sampled; reuse to compare methods on the exact same evidence.
JUDGEMENTS_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/judgements.csv"  # Change this to preserve multiple judgement sets side by side.
JUDGEMENTS_PLOT_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/judgements_plot.csv"  # Change this if you want separate figure-data snapshots.
JUDGEMENTS_FIGURE_STEM = f"{MAIN2_ENV['figure_dir']}/judgements"  # Change this stem to rename the exported figure files.


def generate_triplet_judgements(
    points_df: pd.DataFrame,
    n_judgements: int,
    temperature: float,
    seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    coords = points_df[["x", "y"]].to_numpy(dtype=float)
    rows = []
    for judgement_id in range(n_judgements):
        a_idx, b_idx, c_idx = rng.choice(len(points_df), size=3, replace=False)
        triples = [
            canonical_pair(a_idx, b_idx),
            canonical_pair(a_idx, c_idx),
            canonical_pair(b_idx, c_idx),
        ]
        distances = np.array(
            [
                np.linalg.norm(coords[pair[0]] - coords[pair[1]])
                for pair in triples
            ],
            dtype=float,
        )
        scaled = np.exp(-distances / max(float(temperature), 1e-6))
        probs = scaled / scaled.sum()
        winner_idx = int(rng.choice(len(triples), p=probs))
        winner_pair = triples[winner_idx]
        loser_pairs = [pair for idx, pair in enumerate(triples) if idx != winner_idx]

        rows.append(
            {
                "judgement_id": judgement_id,
                "a_idx": int(a_idx),
                "b_idx": int(b_idx),
                "c_idx": int(c_idx),
                "winner_i": int(winner_pair[0]),
                "winner_j": int(winner_pair[1]),
                "loser1_i": int(loser_pairs[0][0]),
                "loser1_j": int(loser_pairs[0][1]),
                "loser2_i": int(loser_pairs[1][0]),
                "loser2_j": int(loser_pairs[1][1]),
                "distance_winner": float(distances[winner_idx]),
                "prob_winner": float(probs[winner_idx]),
            }
        )
    return pd.DataFrame(rows)


def build_judgement_plot_dataframe(points_df: pd.DataFrame, judgements_df: pd.DataFrame) -> pd.DataFrame:
    gray_scale = ["#2f2f2f", "#767676", "#b6b6b6"]
    palette = sns.color_palette("husl", max(len(judgements_df), 3))
    rows = []

    for row in points_df.itertuples(index=False):
        rows.append(
            {
                "entity_type": "point",
                "judgement_id": -1,
                "point_idx": int(row.point_idx),
                "x": float(row.x),
                "y": float(row.y),
                "x_end": np.nan,
                "y_end": np.nan,
                "marker": row.marker,
                "color": gray_scale[int(row.true_cluster) % len(gray_scale)],
                "line_style": "",
                "alpha": 1.0,
                "linewidth": 1.5,
            }
        )

    coords = points_df.set_index("point_idx")[["x", "y"]]
    for row in judgements_df.itertuples(index=False):
        color = palette[row.judgement_id % len(palette)]
        color_hex = "#{:02x}{:02x}{:02x}".format(
            int(color[0] * 255),
            int(color[1] * 255),
            int(color[2] * 255),
        )
        winner_pair = (int(row.winner_i), int(row.winner_j))
        loser_pairs = [
            (int(row.loser1_i), int(row.loser1_j)),
            (int(row.loser2_i), int(row.loser2_j)),
        ]

        for pair, line_style, alpha, linewidth in [
            (winner_pair, "-", 0.95, 1.8),
            (loser_pairs[0], "--", 0.28, 0.9),
            (loser_pairs[1], "--", 0.28, 0.9),
        ]:
            rows.append(
                {
                    "entity_type": "edge",
                    "judgement_id": int(row.judgement_id),
                    "point_idx": -1,
                    "x": float(coords.loc[pair[0], "x"]),
                    "y": float(coords.loc[pair[0], "y"]),
                    "x_end": float(coords.loc[pair[1], "x"]),
                    "y_end": float(coords.loc[pair[1], "y"]),
                    "marker": "",
                    "color": color_hex,
                    "line_style": line_style,
                    "alpha": alpha,
                    "linewidth": linewidth,
                }
            )

    return pd.DataFrame(rows)


def plot_judgements_from_csv(csv_path: str | Path, stem: str | Path) -> tuple[Path, Path]:
    df = pd.read_csv(csv_path)
    fig, ax = plt.subplots(figsize=(6.4, 5.3))

    edge_df = df[df["entity_type"] == "edge"].copy()
    point_df = df[df["entity_type"] == "point"].copy()

    for row in edge_df.itertuples(index=False):
        ax.plot(
            [row.x, row.x_end],
            [row.y, row.y_end],
            color=row.color,
            linestyle=row.line_style,
            linewidth=float(row.linewidth),
            alpha=float(row.alpha),
            zorder=1,
        )

    for row in point_df.itertuples(index=False):
        ax.scatter(
            row.x,
            row.y,
            facecolors="none",
            edgecolors=row.color,
            marker=row.marker,
            linewidths=1.6,
            s=85,
            zorder=2,
        )

    ax.set_title("Noisy Pairwise Judgements")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    sns.despine(ax=ax)
    fig.tight_layout()
    paths = save_figure_bundle(fig, stem)
    plt.show()
    plt.close(fig)
    return paths


def run_judgement_cell(
    points_csv_path: str | Path,
    n_judgements: int,
    temperature: float,
    seed: int,
    judgements_csv_path: str | Path,
    plot_csv_path: str | Path,
    figure_stem: str | Path,
) -> dict:
    points_df = pd.read_csv(points_csv_path)
    judgements_df = generate_triplet_judgements(points_df, n_judgements, temperature, seed)
    save_dataframe(judgements_df, judgements_csv_path)
    plot_df = build_judgement_plot_dataframe(points_df, judgements_df)
    save_dataframe(plot_df, plot_csv_path)
    plot_judgements_from_csv(plot_csv_path, figure_stem)
    logging.info("Judgement cell completed with %d judgements", len(judgements_df))
    return {
        "judgements_df": judgements_df,
        "judgements_csv_path": str(judgements_csv_path),
        "plot_csv_path": str(plot_csv_path),
        "figure_stem": str(figure_stem),
    }


MAIN2_STATE["judgements"] = run_judgement_cell(
    points_csv_path=MAIN2_STATE["dataset"]["csv_path"],
    n_judgements=JUDGEMENT_COUNT,
    temperature=JUDGEMENT_TEMPERATURE,
    seed=JUDGEMENT_SEED,
    judgements_csv_path=JUDGEMENTS_CSV_PATH,
    plot_csv_path=JUDGEMENTS_PLOT_CSV_PATH,
    figure_stem=JUDGEMENTS_FIGURE_STEM,
)

MAIN2_STATE["judgements"]["judgements_df"].head()


## Cell 4 – Cluster Generation (Max Cut)


In [ ]:
MAXCUT_CLUSTER_COUNT = 3  # Higher asks the cut heuristic to split more aggressively; lower gives coarser partitions.
MAXCUT_RESTARTS = 40  # Higher searches more random assignments; lower runs faster but is less stable.
MAXCUT_MAX_ITER = 250  # Higher lets local search settle more fully; lower stops sooner with rougher cuts.
MAXCUT_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/maxcut_clusters.csv"  # Change this to keep multiple clustering outputs side by side.
MAXCUT_FIGURE_STEM = f"{MAIN2_ENV['figure_dir']}/maxcut_clusters"  # Change this stem to rename the figure bundle.


def build_solid_graph_from_judgements(judgements_df: pd.DataFrame, n_points: int) -> nx.Graph:
    graph = nx.Graph()
    graph.add_nodes_from(range(n_points))
    for row in judgements_df.itertuples(index=False):
        u, v = canonical_pair(row.winner_i, row.winner_j)
        if graph.has_edge(u, v):
            graph[u][v]["weight"] += 1.0
        else:
            graph.add_edge(u, v, weight=1.0)
    return graph


def infer_maxcut_labels_from_judgements(
    judgements_df: pd.DataFrame,
    n_points: int,
    n_clusters: int,
    n_restarts: int,
    max_iter: int,
    seed: int,
) -> np.ndarray:
    graph = build_solid_graph_from_judgements(judgements_df, n_points)
    assignment, _ = greedy_maxcut_partition(graph, n_clusters, n_restarts, max_iter, seed)
    return assignment_dict_to_array(assignment, n_points)


def run_maxcut_cell(
    points_csv_path: str | Path,
    judgements_csv_path: str | Path,
    n_clusters: int,
    n_restarts: int,
    max_iter: int,
    seed: int,
    csv_path: str | Path,
    figure_stem: str | Path,
) -> dict:
    points_df = pd.read_csv(points_csv_path)
    judgements_df = pd.read_csv(judgements_csv_path)
    inferred = infer_maxcut_labels_from_judgements(
        judgements_df=judgements_df,
        n_points=len(points_df),
        n_clusters=n_clusters,
        n_restarts=n_restarts,
        max_iter=max_iter,
        seed=seed,
    )
    export_df = points_df.copy()
    export_df["inferred_cluster"] = inferred
    save_dataframe(export_df, csv_path)
    plot_cluster_scatter_from_csv(csv_path, figure_stem, "Max-Cut Clustering")
    metrics = pairwise_consistency_metrics(export_df["true_cluster"], export_df["inferred_cluster"])
    logging.info("Max-cut metrics: %s", metrics)
    return {
        "clusters_df": export_df,
        "csv_path": str(csv_path),
        "figure_stem": str(figure_stem),
        "metrics": metrics,
    }


MAIN2_STATE["maxcut"] = run_maxcut_cell(
    points_csv_path=MAIN2_STATE["dataset"]["csv_path"],
    judgements_csv_path=MAIN2_STATE["judgements"]["judgements_csv_path"],
    n_clusters=MAXCUT_CLUSTER_COUNT,
    n_restarts=MAXCUT_RESTARTS,
    max_iter=MAXCUT_MAX_ITER,
    seed=SETUP_SEED,
    csv_path=MAXCUT_CSV_PATH,
    figure_stem=MAXCUT_FIGURE_STEM,
)

MAIN2_STATE["maxcut"]["metrics"]


## Cell 5 – Cluster Generation II (Min Cut)


In [ ]:
MINCUT_CLUSTER_COUNT = 3  # Higher forces more partitions from the dashed-edge graph; lower keeps fewer groups.
MINCUT_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/mincut_clusters.csv"  # Change this path to archive different min-cut runs.
MINCUT_FIGURE_STEM = f"{MAIN2_ENV['figure_dir']}/mincut_clusters"  # Change this stem to rename the saved figure bundle.


def build_dashed_graph_from_judgements(judgements_df: pd.DataFrame, n_points: int) -> nx.Graph:
    graph = nx.Graph()
    graph.add_nodes_from(range(n_points))
    for row in judgements_df.itertuples(index=False):
        for pair in [
            canonical_pair(row.loser1_i, row.loser1_j),
            canonical_pair(row.loser2_i, row.loser2_j),
        ]:
            if graph.has_edge(*pair):
                graph[pair[0]][pair[1]]["weight"] += 1.0
            else:
                graph.add_edge(pair[0], pair[1], weight=1.0)
    return graph


def infer_mincut_labels_from_judgements(
    judgements_df: pd.DataFrame,
    n_points: int,
    n_clusters: int,
) -> np.ndarray:
    graph = build_dashed_graph_from_judgements(judgements_df, n_points)
    assignment = hierarchical_mincut_partition(graph, n_clusters)
    return assignment_dict_to_array(assignment, n_points)


def run_mincut_cell(
    points_csv_path: str | Path,
    judgements_csv_path: str | Path,
    n_clusters: int,
    csv_path: str | Path,
    figure_stem: str | Path,
) -> dict:
    points_df = pd.read_csv(points_csv_path)
    judgements_df = pd.read_csv(judgements_csv_path)
    inferred = infer_mincut_labels_from_judgements(judgements_df, len(points_df), n_clusters)
    export_df = points_df.copy()
    export_df["inferred_cluster"] = inferred
    save_dataframe(export_df, csv_path)
    plot_cluster_scatter_from_csv(csv_path, figure_stem, "Min-Cut Clustering", palette_name="muted")
    metrics = pairwise_consistency_metrics(export_df["true_cluster"], export_df["inferred_cluster"])
    logging.info("Min-cut metrics: %s", metrics)
    return {
        "clusters_df": export_df,
        "csv_path": str(csv_path),
        "figure_stem": str(figure_stem),
        "metrics": metrics,
    }


MAIN2_STATE["mincut"] = run_mincut_cell(
    points_csv_path=MAIN2_STATE["dataset"]["csv_path"],
    judgements_csv_path=MAIN2_STATE["judgements"]["judgements_csv_path"],
    n_clusters=MINCUT_CLUSTER_COUNT,
    csv_path=MINCUT_CSV_PATH,
    figure_stem=MINCUT_FIGURE_STEM,
)

MAIN2_STATE["mincut"]["metrics"]


## Cell 6 – Meta Graph


In [ ]:
META_GRAPH_LAYOUT_SEED = 42  # Higher only changes the drawn layout; the graph structure itself stays the same.
META_GRAPH_PLOT_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/meta_graph.csv"  # Change this to preserve alternative graph layouts.
META_GRAPH_FIGURE_STEM = f"{MAIN2_ENV['figure_dir']}/meta_graph"  # Change this stem to rename the exported graph figure.


def build_meta_graph(judgements_df: pd.DataFrame, true_labels: np.ndarray) -> nx.DiGraph:
    graph = nx.DiGraph()

    def same_cluster_for_pair(pair: tuple[int, int]) -> bool:
        return bool(int(true_labels[pair[0]]) == int(true_labels[pair[1]]))

    for row in judgements_df.itertuples(index=False):
        winner = canonical_pair(row.winner_i, row.winner_j)
        loser_pairs = [
            canonical_pair(row.loser1_i, row.loser1_j),
            canonical_pair(row.loser2_i, row.loser2_j),
        ]
        all_pairs = [winner] + loser_pairs
        for pair in all_pairs:
            if pair not in graph:
                graph.add_node(pair, same_cluster=same_cluster_for_pair(pair))
        for loser in loser_pairs:
            graph.add_edge(loser, winner)
    return graph


def export_meta_graph_plot_csv(graph: nx.DiGraph, seed: int, csv_path: str | Path) -> Path:
    positions = nx.spring_layout(graph, seed=seed, k=1.8)
    rows = []
    for node, (x_coord, y_coord) in positions.items():
        rows.append(
            {
                "entity_type": "node",
                "source_i": int(node[0]),
                "source_j": int(node[1]),
                "target_i": -1,
                "target_j": -1,
                "x": float(x_coord),
                "y": float(y_coord),
                "x_end": np.nan,
                "y_end": np.nan,
                "same_cluster": int(graph.nodes[node]["same_cluster"]),
            }
        )
    for source, target in graph.edges():
        rows.append(
            {
                "entity_type": "edge",
                "source_i": int(source[0]),
                "source_j": int(source[1]),
                "target_i": int(target[0]),
                "target_j": int(target[1]),
                "x": float(positions[source][0]),
                "y": float(positions[source][1]),
                "x_end": float(positions[target][0]),
                "y_end": float(positions[target][1]),
                "same_cluster": int(graph.nodes[source]["same_cluster"]),
            }
        )
    df = pd.DataFrame(rows)
    save_dataframe(df, csv_path)
    return Path(csv_path)


def plot_meta_graph_from_csv(csv_path: str | Path, stem: str | Path) -> tuple[Path, Path]:
    df = pd.read_csv(csv_path)
    node_df = df[df["entity_type"] == "node"].copy()
    edge_df = df[df["entity_type"] == "edge"].copy()

    lookup = {
        (int(row.source_i), int(row.source_j)): int(row.same_cluster)
        for row in node_df.itertuples(index=False)
    }

    fig, ax = plt.subplots(figsize=(10, 8))
    for row in edge_df.itertuples(index=False):
        source_same = lookup[(int(row.source_i), int(row.source_j))]
        target_same = lookup[(int(row.target_i), int(row.target_j))]
        color = palette_color(0) if source_same == 1 and target_same == 0 else (0.55, 0.55, 0.55)
        ax.annotate(
            "",
            xy=(row.x_end, row.y_end),
            xytext=(row.x, row.y),
            arrowprops=dict(
                arrowstyle="->",
                color=color,
                lw=1.1,
                alpha=0.75,
                shrinkA=10,
                shrinkB=10,
                connectionstyle="arc3,rad=0.08",
            ),
        )

    for row in node_df.itertuples(index=False):
        color = palette_color(2) if int(row.same_cluster) == 1 else palette_color(3)
        ax.scatter(
            row.x,
            row.y,
            s=420,
            color=color,
            alpha=0.95,
            edgecolors="white",
            linewidths=0.8,
            zorder=3,
        )
        ax.text(
            row.x,
            row.y,
            f"({int(row.source_i)},{int(row.source_j)})",
            ha="center",
            va="center",
            fontsize=6,
            color="white",
            zorder=4,
        )

    ax.set_title("Directed Meta Graph of Pairwise Comparisons")
    ax.axis("off")
    fig.tight_layout()
    paths = save_figure_bundle(fig, stem)
    plt.show()
    plt.close(fig)
    return paths


def run_meta_graph_cell(
    points_csv_path: str | Path,
    judgements_csv_path: str | Path,
    layout_seed: int,
    plot_csv_path: str | Path,
    figure_stem: str | Path,
) -> dict:
    points_df = pd.read_csv(points_csv_path)
    judgements_df = pd.read_csv(judgements_csv_path)
    graph = build_meta_graph(judgements_df, points_df["true_cluster"].to_numpy(dtype=int))
    export_meta_graph_plot_csv(graph, layout_seed, plot_csv_path)
    plot_meta_graph_from_csv(plot_csv_path, figure_stem)
    logging.info("Meta graph cell completed with %d nodes", graph.number_of_nodes())
    return {
        "graph": graph,
        "plot_csv_path": str(plot_csv_path),
        "figure_stem": str(figure_stem),
    }


MAIN2_STATE["meta_graph"] = run_meta_graph_cell(
    points_csv_path=MAIN2_STATE["dataset"]["csv_path"],
    judgements_csv_path=MAIN2_STATE["judgements"]["judgements_csv_path"],
    layout_seed=META_GRAPH_LAYOUT_SEED,
    plot_csv_path=META_GRAPH_PLOT_CSV_PATH,
    figure_stem=META_GRAPH_FIGURE_STEM,
)

(
    MAIN2_STATE["meta_graph"]["graph"].number_of_nodes(),
    MAIN2_STATE["meta_graph"]["graph"].number_of_edges(),
)


## Cell 7 – Meta Graph Optimization


In [ ]:
CONFLICT_CLUSTER_COUNT = 3  # Higher allows more clusters to explain the graph; lower forces broader merges.
CONFLICT_RANDOM_RESTARTS = 12  # Higher improves the greedy discrete search; lower makes this cell faster but noisier.
CONFLICT_SWEEPS = 35  # Higher gives more reassignment passes; lower stops earlier with rougher solutions.
CONFLICT_BALANCE_LAMBDA = 0.7  # Higher punishes collapsed one-cluster solutions more strongly; lower trusts the conflict objective more.
CONFLICT_SEED = 42  # Higher changes the random initialization; keeping it fixed makes comparison with other methods easier.
CONFLICT_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/conflict_graph_clusters.csv"  # Change this to archive several conflict-optimized clusterings.
CONFLICT_FIGURE_STEM = f"{MAIN2_ENV['figure_dir']}/conflict_graph_clusters"  # Change this stem to rename the figure outputs.


def count_conflicting_edges(judgements_df: pd.DataFrame, labels: np.ndarray) -> int:
    conflicts = 0
    for row in judgements_df.itertuples(index=False):
        winner_same = int(labels[int(row.winner_i)] == labels[int(row.winner_j)])
        loser_pairs = [
            (int(row.loser1_i), int(row.loser1_j)),
            (int(row.loser2_i), int(row.loser2_j)),
        ]
        for pair in loser_pairs:
            loser_same = int(labels[pair[0]] == labels[pair[1]])
            if loser_same == 1 and winner_same == 0:
                conflicts += 1
    return conflicts


def cluster_balance_penalty(labels: np.ndarray, n_clusters: int) -> float:
    counts = np.bincount(labels.astype(int), minlength=n_clusters).astype(float)
    target = len(labels) / float(n_clusters)
    return float(np.mean((counts - target) ** 2) / max(target, 1.0))


def optimize_meta_graph_clustering(
    judgements_df: pd.DataFrame,
    n_points: int,
    n_clusters: int,
    n_restarts: int,
    n_sweeps: int,
    balance_lambda: float,
    seed: int,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    best_labels = None
    best_objective = np.inf

    def objective(candidate: np.ndarray) -> float:
        return count_conflicting_edges(judgements_df, candidate) + balance_lambda * cluster_balance_penalty(candidate, n_clusters)

    for _ in range(n_restarts):
        labels = rng.integers(0, n_clusters, size=n_points, endpoint=False)
        for _ in range(n_sweeps):
            improved = False
            order = np.arange(n_points)
            rng.shuffle(order)
            for point_idx in order:
                current_value = objective(labels)
                current_cluster = labels[point_idx]
                best_local_cluster = current_cluster
                best_local_value = current_value
                for candidate_cluster in range(n_clusters):
                    if candidate_cluster == current_cluster:
                        continue
                    labels[point_idx] = candidate_cluster
                    candidate_value = objective(labels)
                    if candidate_value < best_local_value:
                        best_local_value = candidate_value
                        best_local_cluster = candidate_cluster
                    labels[point_idx] = current_cluster
                if best_local_cluster != current_cluster:
                    labels[point_idx] = best_local_cluster
                    improved = True
            if not improved:
                break
        value = objective(labels)
        if value < best_objective:
            best_objective = value
            best_labels = labels.copy()

    return np.asarray(best_labels, dtype=int)


def run_conflict_optimization_cell(
    points_csv_path: str | Path,
    judgements_csv_path: str | Path,
    n_clusters: int,
    n_restarts: int,
    n_sweeps: int,
    balance_lambda: float,
    seed: int,
    csv_path: str | Path,
    figure_stem: str | Path,
) -> dict:
    points_df = pd.read_csv(points_csv_path)
    judgements_df = pd.read_csv(judgements_csv_path)
    labels = optimize_meta_graph_clustering(
        judgements_df=judgements_df,
        n_points=len(points_df),
        n_clusters=n_clusters,
        n_restarts=n_restarts,
        n_sweeps=n_sweeps,
        balance_lambda=balance_lambda,
        seed=seed,
    )
    export_df = points_df.copy()
    export_df["inferred_cluster"] = labels
    save_dataframe(export_df, csv_path)
    plot_cluster_scatter_from_csv(csv_path, figure_stem, "Conflict-Graph Optimization", palette_name="bright")
    metrics = pairwise_consistency_metrics(export_df["true_cluster"], export_df["inferred_cluster"])
    metrics["conflicting_edges"] = count_conflicting_edges(judgements_df, labels)
    logging.info("Conflict-graph optimization metrics: %s", metrics)
    return {
        "clusters_df": export_df,
        "csv_path": str(csv_path),
        "figure_stem": str(figure_stem),
        "metrics": metrics,
    }


MAIN2_STATE["conflict_graph"] = run_conflict_optimization_cell(
    points_csv_path=MAIN2_STATE["dataset"]["csv_path"],
    judgements_csv_path=MAIN2_STATE["judgements"]["judgements_csv_path"],
    n_clusters=CONFLICT_CLUSTER_COUNT,
    n_restarts=CONFLICT_RANDOM_RESTARTS,
    n_sweeps=CONFLICT_SWEEPS,
    balance_lambda=CONFLICT_BALANCE_LAMBDA,
    seed=CONFLICT_SEED,
    csv_path=CONFLICT_CSV_PATH,
    figure_stem=CONFLICT_FIGURE_STEM,
)

MAIN2_STATE["conflict_graph"]["metrics"]


## Cell 8 – Learn Embedding


In [ ]:
EMBEDDING_DIM = 2  # Higher allows richer latent geometry; lower forces more compression but is easier to visualize.
EMBEDDING_LR = 0.05  # Higher converges faster but can oscillate; lower is steadier but slower.
EMBEDDING_EPOCHS = 1200  # Higher fits the likelihood more closely; lower is faster but may underfit the judgements.
EMBEDDING_BATCH_SIZE = 16  # Higher gives smoother gradients; lower gives noisier updates and more optimizer steps.
EMBEDDING_SEED = 42  # Higher changes the random initial positions; reuse to compare runs consistently.
EMBEDDING_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/embedding_comparison.csv"  # Change this to archive multiple embedding results.
EMBEDDING_FIGURE_STEM = f"{MAIN2_ENV['figure_dir']}/embedding_comparison"  # Change this stem to rename the figure exports.


def bradley_terry_log_likelihood(
    embedding_tensor: torch.Tensor,
    winner_pairs: torch.Tensor,
    loser_pairs_1: torch.Tensor,
    loser_pairs_2: torch.Tensor,
) -> torch.Tensor:
    def pair_distance(pair_tensor: torch.Tensor) -> torch.Tensor:
        return torch.norm(embedding_tensor[pair_tensor[:, 0]] - embedding_tensor[pair_tensor[:, 1]], dim=-1)

    d_w = pair_distance(winner_pairs)
    d_l1 = pair_distance(loser_pairs_1)
    d_l2 = pair_distance(loser_pairs_2)
    logits = torch.stack([-d_w, -d_l1, -d_l2], dim=-1)
    return (-d_w - torch.logsumexp(logits, dim=-1)).mean()


def learn_embedding_from_judgements(
    points_df: pd.DataFrame,
    judgements_df: pd.DataFrame,
    dim: int,
    lr: float,
    epochs: int,
    batch_size: int,
    seed: int,
    device: str,
    dtype: torch.dtype,
) -> tuple[np.ndarray, list[float]]:
    seed_everything(seed)
    winner_pairs = torch.tensor(
        judgements_df[["winner_i", "winner_j"]].to_numpy(dtype=np.int64),
        dtype=torch.long,
        device=device,
    )
    loser_pairs_1 = torch.tensor(
        judgements_df[["loser1_i", "loser1_j"]].to_numpy(dtype=np.int64),
        dtype=torch.long,
        device=device,
    )
    loser_pairs_2 = torch.tensor(
        judgements_df[["loser2_i", "loser2_j"]].to_numpy(dtype=np.int64),
        dtype=torch.long,
        device=device,
    )

    embedding = nn.Parameter(torch.randn(len(points_df), dim, dtype=dtype, device=device))
    optimizer = optim.Adam([embedding], lr=lr)
    losses = []

    n_obs = len(judgements_df)
    for _ in range(epochs):
        order = torch.randperm(n_obs, device=device)
        epoch_losses = []
        for start in range(0, n_obs, batch_size):
            idx = order[start:start + batch_size]
            optimizer.zero_grad()
            log_likelihood = bradley_terry_log_likelihood(
                embedding,
                winner_pairs[idx],
                loser_pairs_1[idx],
                loser_pairs_2[idx],
            )
            loss = -log_likelihood
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))
        losses.append(float(np.mean(epoch_losses)))

    learned = embedding.detach().cpu().numpy()
    gt_coords = points_df[["x", "y"]].to_numpy(dtype=float)
    centered_gt = gt_coords - gt_coords.mean(axis=0)
    centered_learned = learned - learned.mean(axis=0)
    rotation, _ = orthogonal_procrustes(centered_learned, centered_gt)
    aligned = centered_learned @ rotation
    return aligned, losses


def export_embedding_comparison_csv(
    points_df: pd.DataFrame,
    learned_embedding: np.ndarray,
    csv_path: str | Path,
) -> Path:
    rows = []
    gt_coords = points_df[["x", "y"]].to_numpy(dtype=float)
    for row in points_df.itertuples(index=False):
        learned_x, learned_y = learned_embedding[int(row.point_idx)]
        rows.append(
            {
                "panel": "learned",
                "point_idx": int(row.point_idx),
                "x": float(learned_x),
                "y": float(learned_y),
                "true_cluster": int(row.true_cluster),
                "marker": row.marker,
            }
        )
        rows.append(
            {
                "panel": "ground_truth",
                "point_idx": int(row.point_idx),
                "x": float(gt_coords[int(row.point_idx), 0]),
                "y": float(gt_coords[int(row.point_idx), 1]),
                "true_cluster": int(row.true_cluster),
                "marker": row.marker,
            }
        )
    df = pd.DataFrame(rows)
    save_dataframe(df, csv_path)
    return Path(csv_path)


def plot_embedding_comparison_from_csv(csv_path: str | Path, stem: str | Path) -> tuple[Path, Path]:
    df = pd.read_csv(csv_path)
    learned_df = df[df["panel"] == "learned"].copy().sort_values("point_idx")
    gt_df = df[df["panel"] == "ground_truth"].copy().sort_values("point_idx")
    palette = sns.color_palette("deep", max(int(df["true_cluster"].max()) + 1, 3))

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.6, 4.7))
    for cluster_id in sorted(df["true_cluster"].unique()):
        mask_left = learned_df["true_cluster"] == cluster_id
        mask_right = gt_df["true_cluster"] == cluster_id
        ax_left.scatter(
            learned_df.loc[mask_left, "x"],
            learned_df.loc[mask_left, "y"],
            facecolors="none",
            edgecolors=[palette[int(cluster_id) % len(palette)]],
            marker=marker_for_cluster(cluster_id),
            linewidths=1.8,
            s=90,
            label=f"Cluster {int(cluster_id)}",
        )
        ax_right.scatter(
            gt_df.loc[mask_right, "x"],
            gt_df.loc[mask_right, "y"],
            facecolors="none",
            edgecolors=[palette[int(cluster_id) % len(palette)]],
            marker=marker_for_cluster(cluster_id),
            linewidths=1.8,
            s=90,
            label=f"Cluster {int(cluster_id)}",
        )

    for left_row, right_row in zip(learned_df.itertuples(index=False), gt_df.itertuples(index=False)):
        fig.add_artist(
            ConnectionPatch(
                xyA=(left_row.x, left_row.y),
                coordsA=ax_left.transData,
                xyB=(right_row.x, right_row.y),
                coordsB=ax_right.transData,
                color="gray",
                alpha=0.18,
                linewidth=0.55,
            )
        )

    ax_left.set_title("Learned Embedding")
    ax_right.set_title("Ground Truth Embedding")
    for ax in (ax_left, ax_right):
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.legend(frameon=False, fontsize=8)
        sns.despine(ax=ax)

    fig.tight_layout()
    paths = save_figure_bundle(fig, stem)
    plt.show()
    plt.close(fig)
    return paths


def run_embedding_cell(
    points_csv_path: str | Path,
    judgements_csv_path: str | Path,
    dim: int,
    lr: float,
    epochs: int,
    batch_size: int,
    seed: int,
    csv_path: str | Path,
    figure_stem: str | Path,
) -> dict:
    points_df = pd.read_csv(points_csv_path)
    judgements_df = pd.read_csv(judgements_csv_path)
    learned_embedding, losses = learn_embedding_from_judgements(
        points_df=points_df,
        judgements_df=judgements_df,
        dim=dim,
        lr=lr,
        epochs=epochs,
        batch_size=batch_size,
        seed=seed,
        device=MAIN2_ENV["device"],
        dtype=MAIN2_ENV["dtype"],
    )
    export_embedding_comparison_csv(points_df, learned_embedding, csv_path)
    plot_embedding_comparison_from_csv(csv_path, figure_stem)
    embedding_df = pd.read_csv(csv_path)
    learned_points_df = embedding_df[embedding_df["panel"] == "learned"].copy().reset_index(drop=True)
    logging.info("Embedding cell completed. Final loss %.4f", losses[-1])
    return {
        "embedding_df": embedding_df,
        "learned_points_df": learned_points_df,
        "csv_path": str(csv_path),
        "figure_stem": str(figure_stem),
        "losses": losses,
    }


MAIN2_STATE["embedding"] = run_embedding_cell(
    points_csv_path=MAIN2_STATE["dataset"]["csv_path"],
    judgements_csv_path=MAIN2_STATE["judgements"]["judgements_csv_path"],
    dim=EMBEDDING_DIM,
    lr=EMBEDDING_LR,
    epochs=EMBEDDING_EPOCHS,
    batch_size=EMBEDDING_BATCH_SIZE,
    seed=EMBEDDING_SEED,
    csv_path=EMBEDDING_CSV_PATH,
    figure_stem=EMBEDDING_FIGURE_STEM,
)

MAIN2_STATE["embedding"]["losses"][-5:]


## Cell 9 – KNN Learn Embedding


In [ ]:
KNN_NEIGHBORS = 5  # Higher smooths the graph over more neighbors; lower keeps the graph more local and fragmented.
KNN_CLUSTER_COUNT = 3  # Higher asks the KNN graph partitioner for more groups; lower merges broader neighborhoods.
KNN_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/knn_embedding_clusters.csv"  # Change this to archive multiple KNN-clustering outputs.
KNN_FIGURE_STEM = f"{MAIN2_ENV['figure_dir']}/knn_embedding_clusters"  # Change this stem to rename the KNN figure bundle.


def build_knn_similarity_graph(embedding_xy: np.ndarray, n_neighbors: int) -> nx.Graph:
    graph = nx.Graph()
    n_points = embedding_xy.shape[0]
    graph.add_nodes_from(range(n_points))
    distances = np.linalg.norm(embedding_xy[:, None, :] - embedding_xy[None, :, :], axis=-1)
    for idx in range(n_points):
        neighbor_order = np.argsort(distances[idx])[1:n_neighbors + 1]
        for neighbor in neighbor_order:
            weight = 1.0 / max(float(distances[idx, neighbor]), 1e-6)
            if graph.has_edge(idx, int(neighbor)):
                graph[idx][int(neighbor)]["weight"] = max(graph[idx][int(neighbor)]["weight"], weight)
            else:
                graph.add_edge(idx, int(neighbor), weight=weight)
    return graph


def cluster_knn_embedding(
    learned_points_df: pd.DataFrame,
    n_neighbors: int,
    n_clusters: int,
) -> np.ndarray:
    embedding_xy = learned_points_df[["x", "y"]].to_numpy(dtype=float)
    graph = build_knn_similarity_graph(embedding_xy, n_neighbors)
    assignment = hierarchical_mincut_partition(graph, n_clusters)
    return assignment_dict_to_array(assignment, len(learned_points_df))


def run_knn_cell(
    points_csv_path: str | Path,
    embedding_csv_path: str | Path,
    n_neighbors: int,
    n_clusters: int,
    csv_path: str | Path,
    figure_stem: str | Path,
) -> dict:
    points_df = pd.read_csv(points_csv_path)
    embedding_df = pd.read_csv(embedding_csv_path)
    learned_points_df = embedding_df[embedding_df["panel"] == "learned"].copy().sort_values("point_idx")
    inferred = cluster_knn_embedding(learned_points_df, n_neighbors, n_clusters)

    export_df = learned_points_df[["point_idx", "x", "y", "true_cluster", "marker"]].copy()
    export_df["inferred_cluster"] = inferred
    save_dataframe(export_df, csv_path)
    plot_cluster_scatter_from_csv(csv_path, figure_stem, "KNN Graph Clustering on Learned Embedding", palette_name="tab10")
    metrics = pairwise_consistency_metrics(points_df["true_cluster"], export_df["inferred_cluster"])
    logging.info("KNN-embedding metrics: %s", metrics)
    return {
        "clusters_df": export_df,
        "csv_path": str(csv_path),
        "figure_stem": str(figure_stem),
        "metrics": metrics,
    }


MAIN2_STATE["knn_embedding"] = run_knn_cell(
    points_csv_path=MAIN2_STATE["dataset"]["csv_path"],
    embedding_csv_path=MAIN2_STATE["embedding"]["csv_path"],
    n_neighbors=KNN_NEIGHBORS,
    n_clusters=KNN_CLUSTER_COUNT,
    csv_path=KNN_CSV_PATH,
    figure_stem=KNN_FIGURE_STEM,
)

MAIN2_STATE["knn_embedding"]["metrics"]


## Cell 10 – Comparison of All Methods


In [ ]:
EVAL_RUNS = 20  # Higher gives smoother confidence intervals; lower makes this benchmark much faster.
EVAL_MIN_CLUSTERS = 2  # Lower narrows the task family; higher minimum removes the easiest cases.
EVAL_MAX_CLUSTERS = 5  # Higher broadens task difficulty; lower caps structural variety.
EVAL_POINTS = 50  # Higher makes every run richer but slower; lower reduces pairwise evaluation cost.
EVAL_QUERIES = 1000  # Higher gives more evidence per run; lower makes the methods compete under stronger data scarcity.
EVAL_PREFIX_SIZES = list(range(100, 1001, 100))  # Finer steps give denser curves; coarser steps shorten the benchmark table.
EVAL_MAXCUT_RESTARTS = 8  # Higher stabilizes max-cut evaluation; lower keeps the benchmark lighter.
EVAL_MAXCUT_MAX_ITER = 80  # Higher lets the local max-cut search settle more; lower trades accuracy for speed.
EVAL_CONFLICT_RESTARTS = 5  # Higher helps the discrete conflict optimizer escape bad initials; lower is faster.
EVAL_CONFLICT_SWEEPS = 15  # Higher lets labels re-balance longer; lower stops sooner.
EVAL_CONFLICT_BALANCE_LAMBDA = 0.9  # Higher resists collapsed clusterings; lower focuses more tightly on conflict count.
EVAL_EMBED_EPOCHS = 80  # Higher improves the embedding fit; lower keeps the repeated benchmark practical.
EVAL_EMBED_BATCH_SIZE = 64  # Higher gives smoother but fewer updates; lower gives noisier but more frequent gradient steps.
EVAL_KNN_NEIGHBORS = 6  # Higher smooths the learned-embedding graph; lower makes KNN clustering more local.
EVAL_RESULTS_CSV_PATH = f"{MAIN2_ENV['figure_dir']}/evaluation_results.csv"  # Change this to keep multiple benchmark snapshots.
EVAL_FIGURE_STEM = f"{MAIN2_ENV['figure_dir']}/evaluation_balanced_accuracy"  # Change this stem to rename the summary figure bundle.


def allocate_cluster_sizes(n_points: int, n_clusters: int) -> list[int]:
    base = n_points // n_clusters
    sizes = [base] * n_clusters
    for idx in range(n_points - base * n_clusters):
        sizes[idx] += 1
    return sizes


def sample_cluster_params_for_evaluation(n_clusters: int, rng: np.random.Generator) -> list[dict]:
    angles = np.linspace(0.0, 2.0 * np.pi, num=n_clusters, endpoint=False)
    radius = rng.uniform(2.0, 6.0)
    params = []
    for cluster_id, angle in enumerate(angles):
        params.append(
            {
                "mean_x": float(radius * np.cos(angle) + rng.normal(scale=0.5)),
                "mean_y": float(radius * np.sin(angle) + rng.normal(scale=0.5)),
                "std": float(rng.uniform(0.25, 0.9)),
                "label": cluster_id,
            }
        )
    return params


def generate_evaluation_dataset(n_points: int, n_clusters: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    cluster_params = sample_cluster_params_for_evaluation(n_clusters, rng)
    sizes = allocate_cluster_sizes(n_points, n_clusters)
    rows = []
    point_idx = 0
    for spec, cluster_size in zip(cluster_params, sizes):
        mean = np.array([spec["mean_x"], spec["mean_y"]], dtype=float)
        samples = rng.normal(loc=mean, scale=float(spec["std"]), size=(cluster_size, 2))
        for x_coord, y_coord in samples:
            rows.append(
                {
                    "point_idx": point_idx,
                    "x": float(x_coord),
                    "y": float(y_coord),
                    "true_cluster": int(spec["label"]),
                    "marker": marker_for_cluster(int(spec["label"])),
                }
            )
            point_idx += 1
    return pd.DataFrame(rows)


def evaluate_single_method(
    method_name: str,
    points_df: pd.DataFrame,
    judgements_df: pd.DataFrame,
    n_clusters: int,
    seed: int,
) -> np.ndarray:
    if method_name == "maxcut":
        return infer_maxcut_labels_from_judgements(
            judgements_df,
            n_points=len(points_df),
            n_clusters=n_clusters,
            n_restarts=EVAL_MAXCUT_RESTARTS,
            max_iter=EVAL_MAXCUT_MAX_ITER,
            seed=seed,
        )
    if method_name == "mincut":
        return infer_mincut_labels_from_judgements(
            judgements_df,
            n_points=len(points_df),
            n_clusters=n_clusters,
        )
    if method_name == "conflict_graph":
        return optimize_meta_graph_clustering(
            judgements_df=judgements_df,
            n_points=len(points_df),
            n_clusters=n_clusters,
            n_restarts=EVAL_CONFLICT_RESTARTS,
            n_sweeps=EVAL_CONFLICT_SWEEPS,
            balance_lambda=EVAL_CONFLICT_BALANCE_LAMBDA,
            seed=seed,
        )
    if method_name == "knn_embedding":
        learned_embedding, _ = learn_embedding_from_judgements(
            points_df=points_df,
            judgements_df=judgements_df,
            dim=2,
            lr=EMBEDDING_LR,
            epochs=EVAL_EMBED_EPOCHS,
            batch_size=EVAL_EMBED_BATCH_SIZE,
            seed=seed,
            device=MAIN2_ENV["device"],
            dtype=MAIN2_ENV["dtype"],
        )
        learned_df = points_df[["point_idx", "true_cluster", "marker"]].copy()
        learned_df["x"] = learned_embedding[:, 0]
        learned_df["y"] = learned_embedding[:, 1]
        return cluster_knn_embedding(learned_df, EVAL_KNN_NEIGHBORS, n_clusters)
    raise ValueError(f"Unknown method: {method_name}")


def run_evaluation_benchmark(
    n_runs: int,
    min_clusters: int,
    max_clusters: int,
    n_points: int,
    n_queries: int,
    prefix_sizes: list[int],
    csv_path: str | Path,
    seed: int,
) -> pd.DataFrame:
    rows = []
    methods = ["maxcut", "mincut", "conflict_graph", "knn_embedding"]
    for run_id in range(n_runs):
        run_seed = seed + 1000 * run_id
        rng = np.random.default_rng(run_seed)
        n_clusters = int(rng.integers(min_clusters, max_clusters + 1))
        points_df = generate_evaluation_dataset(n_points, n_clusters, run_seed)
        judgements_df = generate_triplet_judgements(
            points_df=points_df,
            n_judgements=n_queries,
            temperature=1.0,
            seed=run_seed + 1,
        )
        for prefix_size in prefix_sizes:
            subset = judgements_df.head(prefix_size).reset_index(drop=True)
            for method_name in methods:
                predicted_labels = evaluate_single_method(
                    method_name=method_name,
                    points_df=points_df,
                    judgements_df=subset,
                    n_clusters=n_clusters,
                    seed=run_seed + prefix_size,
                )
                metrics = pairwise_consistency_metrics(points_df["true_cluster"], predicted_labels)
                rows.append(
                    {
                        "run_id": run_id,
                        "n_clusters": n_clusters,
                        "n_samples": prefix_size,
                        "method": method_name,
                        "pair_accuracy": metrics["pair_accuracy"],
                        "balanced_accuracy": metrics["balanced_accuracy"],
                        "true_positive_rate": metrics["true_positive_rate"],
                        "true_negative_rate": metrics["true_negative_rate"],
                    }
                )
    df = pd.DataFrame(rows)
    save_dataframe(df, csv_path)
    return df


def plot_evaluation_from_csv(csv_path: str | Path, stem: str | Path) -> tuple[Path, Path]:
    df = pd.read_csv(csv_path)
    fig, ax = plt.subplots(figsize=(8.2, 5.0))
    sns.lineplot(
        data=df,
        x="n_samples",
        y="balanced_accuracy",
        hue="method",
        style="method",
        estimator="mean",
        errorbar=("ci", 95),
        n_boot=300,
        marker="o",
        ax=ax,
    )
    ax.set_title("Balanced Accuracy vs. Number of Queried Comparisons")
    ax.set_xlabel("Number of judged triplets used")
    ax.set_ylabel("Balanced accuracy")
    ax.legend(frameon=False, title="Method")
    sns.despine(ax=ax)
    fig.tight_layout()
    paths = save_figure_bundle(fig, stem)
    plt.show()
    plt.close(fig)
    return paths


def run_evaluation_cell(
    n_runs: int,
    min_clusters: int,
    max_clusters: int,
    n_points: int,
    n_queries: int,
    prefix_sizes: list[int],
    csv_path: str | Path,
    figure_stem: str | Path,
    seed: int,
) -> dict:
    results_df = run_evaluation_benchmark(
        n_runs=n_runs,
        min_clusters=min_clusters,
        max_clusters=max_clusters,
        n_points=n_points,
        n_queries=n_queries,
        prefix_sizes=prefix_sizes,
        csv_path=csv_path,
        seed=seed,
    )
    plot_evaluation_from_csv(csv_path, figure_stem)
    summary_df = (
        results_df.groupby(["method", "n_samples"], as_index=False)
        .agg(
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            pair_accuracy_mean=("pair_accuracy", "mean"),
        )
    )
    logging.info("Evaluation benchmark completed with %d rows", len(results_df))
    return {
        "results_df": results_df,
        "summary_df": summary_df,
        "csv_path": str(csv_path),
        "figure_stem": str(figure_stem),
    }


MAIN2_STATE["evaluation"] = run_evaluation_cell(
    n_runs=EVAL_RUNS,
    min_clusters=EVAL_MIN_CLUSTERS,
    max_clusters=EVAL_MAX_CLUSTERS,
    n_points=EVAL_POINTS,
    n_queries=EVAL_QUERIES,
    prefix_sizes=EVAL_PREFIX_SIZES,
    csv_path=EVAL_RESULTS_CSV_PATH,
    figure_stem=EVAL_FIGURE_STEM,
    seed=SETUP_SEED,
)

MAIN2_STATE["evaluation"]["summary_df"].head(12)
